<h1>Group 34 - Assignment 2</h1>
<h3>Dora Khoshimova, Sevval Yelmer </h3>

<h1> Random Forest Model </h1>

<h2>1. Installing Libraries</h2>

In [141]:
!pip install pandas numpy scikit-learn shap lime opencv-python scikit-image matplotlib
!pip install kagglehub pandas opencv-python numpy scikit-learn tensorflow
!pip install torch torchvision torchaudio
!pip install interpret


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [142]:
# General Dependecies
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
import kagglehub

# Dependecies for Random Forest Classifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report


# Dependecies for Explanaible Algorithms
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
import shap
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

<h2>2. Dataset prep </h2>

In [143]:
# Calling the path and joining it to a csv file
path = kagglehub.dataset_download("faysalmiah1721758/breast-cancer-data")
csv_path = os.path.join(path, "breast-cancer-data.csv")
data = pd.read_csv(csv_path)

# Printing information to see if we were able to load the dataset correctly
#print(data.head)
# data.info()
# data.shape 
# data.describe(include="object")

data.isnull().sum() # these are the missing values, they will be removed later


age            0
menopause      0
tumer-size     0
inv-nodes      0
node-caps      8
deg-malig      0
breast         0
breast-quad    1
irradiate      0
class          0
dtype: int64

<h3>a. Dropping the missing values</h3>

In [ ]:
# To ensure that we do not have any missing values, we decided to remove them

data.dropna(subset=["node-caps", "breast-quad"], inplace=True)
data.isnull().sum() # now all of them is supposed to be a 0 because we dropped the missing values


age            0
menopause      0
tumer-size     0
inv-nodes      0
node-caps      0
deg-malig      0
breast         0
breast-quad    0
irradiate      0
class          0
dtype: int64

<h3>b. Label Encoding</h3>

In [ ]:
# Since the dataset had different values such as strings, float and integers, we decided to turn everything into int
label_encoder = LabelEncoder()
data["age"] = label_encoder.fit_transform(data["age"])
data["menopause"] = label_encoder.fit_transform(data["menopause"])
data["tumer-size"] = label_encoder.fit_transform(data["tumer-size"])
data["inv-nodes"] = label_encoder.fit_transform(data["inv-nodes"])
data["node-caps"] = label_encoder.fit_transform(data["node-caps"])
data["deg-malig"] = label_encoder.fit_transform(data["deg-malig"])
data["breast"] = label_encoder.fit_transform(data["breast"])
data["breast-quad"] = label_encoder.fit_transform(data["breast-quad"])
data["irradiate"] = label_encoder.fit_transform(data["irradiate"])
data["class"] = label_encoder.fit_transform(data["class"])

data.head


<bound method NDFrame.head of      age  menopause  tumer-size  inv-nodes  node-caps  deg-malig  breast  \
0      2          2           2          0          1          2       1   
1      3          0           2          0          0          0       1   
2      3          0           6          0          0          1       0   
3      2          2           6          0          1          2       1   
4      2          2           5          4          1          1       0   
..   ...        ...         ...        ...        ...        ...     ...   
281    3          0           5          5          1          1       0   
282    3          2           4          4          1          1       0   
283    1          2           5          5          1          1       1   
284    3          2           2          0          0          1       1   
285    3          0           7          0          0          2       0   

     breast-quad  irradiate  class  
0              2    

<h3>c. Splitting the data </h3>

In [ ]:
X = data.drop("class", axis=1)
y = data["class"]

X = pd.get_dummies(X, drop_first=True)


In [ ]:
# Splitting the data in 30/70 ratio, we decided to go wth 30/70 instead of 20/80 because the dataset is not balanced
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=9)

In [134]:
"""from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

gr_space = {
    "max_depth": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "n_estimators": [100, 200, 300, 400, 500],
    "max_features": [2, 4, 6, 8, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf = RandomForestClassifier(class_weight="balanced", random_state=42)

grid_search = GridSearchCV(
    rf, param_grid=gr_space, cv=5, scoring="f1_macro", verbose=3, n_jobs=-1
)

model_grid = grid_search.fit(X_train, y_train)

print("Best hyperparameters are " + str(model_grid.best_params_))
print("Best score is: " + str(model_grid.best_score_))
"""

'from sklearn.model_selection import GridSearchCV\nfrom sklearn.ensemble import RandomForestClassifier\n\ngr_space = {\n    "max_depth": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],\n    "n_estimators": [100, 200, 300, 400, 500],\n    "max_features": [2, 4, 6, 8, 10],\n    "min_samples_leaf": [1, 2, 4],\n}\n\nrf = RandomForestClassifier(class_weight="balanced", random_state=42)\n\ngrid_search = GridSearchCV(\n    rf, param_grid=gr_space, cv=5, scoring="f1_macro", verbose=3, n_jobs=-1\n)\n\nmodel_grid = grid_search.fit(X_train, y_train)\n\nprint("Best hyperparameters are " + str(model_grid.best_params_))\nprint("Best score is: " + str(model_grid.best_score_))\n'

<h2>3. Running the Random Forest model </h2>

In [ ]:
model = RandomForestClassifier(
    n_estimators=300, max_depth=8, class_weight="balanced", random_state=42
)
model.fit(X_train, y_train)


,n_estimators,300
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


<h2> Analyzing the outcome of the model </h2>

In [ ]:
y_pred = model.predict(X_test)

#print(y_pred)

print(f"Accuracy score is: {accuracy_score(y_test, y_pred)}")
print(f'F1 score is: {f1_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

Accuracy score is: 0.7906976744186046
F1 score is: 0.55
              precision    recall  f1-score   support

           0       0.84      0.89      0.86        64
           1       0.61      0.50      0.55        22

    accuracy                           0.79        86
   macro avg       0.72      0.70      0.71        86
weighted avg       0.78      0.79      0.78        86



<h1> Explainable Boosting Classifier </h1>

In [ ]:
ebm = ExplainableBoostingClassifier(random_state=42)

ebm.fit(X_train, y_train)

y_pred_ebm = ebm.predict(X_test)
print(f"EBM Accuracy: {accuracy_score(y_test, y_pred_ebm)}")
print(classification_report(y_test, y_pred_ebm))


EBM Accuracy: 0.7325581395348837
              precision    recall  f1-score   support

           0       0.81      0.84      0.82        64
           1       0.47      0.41      0.44        22

    accuracy                           0.73        86
   macro avg       0.64      0.63      0.63        86
weighted avg       0.72      0.73      0.73        86



In [ ]:
ebm_global = ebm.explain_global()
show(ebm_global)
ebm_local = ebm.explain_local(X_test[:5], y_test[:5])
show(ebm_local)


<!-- http://127.0.0.1:7001/5231815040/ -->

<!-- http://127.0.0.1:7001/5231811152/ -->